# SQL Worksheet — Week2

Use the following tables from the Riva Data Platform:

- `rivadataplatform.dataproduct.dim_batch`
- `rivadataplatform.dataproduct.dim_class`
- `rivadataplatform.dataproduct.fact_attendance`
- `rivadataplatform.dataproduct.dim_student`
- `rivadataplatform.dataproduct.dim_date`

**Instructions**
- Write SQL for each question.
- Do not modify the source data.
- Use clear aliases where JOINs are involved.
- Unless a question specifically asks for a particular column, select only the columns needed to answer it.


## Tables / Relationships

Useful keys:
- `dim_student.student_key` ↔ `fact_attendance.student_key`
- `dim_class.class_key` ↔ `fact_attendance.class_key`
- `dim_batch.batch_key` ↔ `fact_attendance.batch_key`
- `dim_class.batch_id` ↔ `dim_batch.batch_id`

## Question 1 — Attendance by Student Location
Build the query in these steps:

**1.1 — Select student location**
From `dim_student`, select the city and replace null city values with `Unknown city`.

**1.2 — Join attendance**
Join `fact_attendance` using `student_key` and add the attendance record identifier and status.

**1.3 — Add class and batch dimensions**
Join `dim_class` using `class_key` and `dim_batch` using `batch_key`.

**1.4 — Group and aggregate**
Group by city and batch name. Count distinct students, total attendance records, and `Late` or `Absent` issue records.

**1.5 — Filter and sort**
Keep only groups with at least one issue and order by issue records descending, then city.

Use readable labels for null batch names.

In [0]:
--Write your code here
SELECT 
    student_name,
    CASE
        WHEN city = 'nan' THEN 'Unknown city'
        ELSE city
    END AS city
FROM rivadataplatform.dataproduct.dim_students;


In [0]:
--Write your code here
--Join attendance Join fact_attendance using student_key and add the attendance record identifier and status.
SELECT
    s.student_name,
    a.attendance_id,
    a.attendance_status
FROM rivadataplatform.dataproduct.dim_students AS s
JOIN rivadataplatform.dataproduct.fact_attendance AS a
    ON s.student_key = a.student_key;

In [0]:
--Add class and batch dimensions Join dim_class using class_key and dim_batch using batch_key
select
    
    s.student_id,
    a.attendance_id,
    a.attendance_status,
    c.class_id,
    b.batch_id
from rivadataplatform.dataproduct.dim_students as s

join rivadataplatform.dataproduct.fact_attendance as a
    on s.student_key = a.student_key

join rivadataplatform.dataproduct.dim_class as c
    on a.class_key = c.class_key

join rivadataplatform.dataproduct.dim_batch as b
    on a.batch_key = b.batch_key


In [0]:
--Group and aggregate Group by city and batch name. Count distinct students, total attendance records, and Late or Absent issue records.

SELECT
    s.city,
    b.batch_name,
    COUNT(DISTINCT s.student_id) AS student_count,
    COUNT(a.attendance_id) AS attendance_count,
    COUNT(CASE WHEN a.attendance_status IN ('Late', 'Absent') THEN 1 END) AS late_absent_count
FROM rivadataplatform.dataproduct.dim_students AS s

JOIN rivadataplatform.dataproduct.fact_attendance AS a
    ON s.student_key = a.student_key


JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON a.batch_key = b.batch_key

GROUP BY
    s.city,
    b.batch_name;    

## Question 2 — Class Calendar Status Counts
Build the query in these steps:

**2.1 — Select class calendar columns**
From `dim_class`, select class date, class day, topic, and class key. Replace null topics with `Topic not assigned`.

**2.2 — Join attendance**
Join `fact_attendance` using `class_key` and add the attendance record identifier and status.

**2.3 — Add batch and date details**
Join `dim_batch` using `batch_key` and `dim_date` using `date_key`. Use the date dimension day name when available.

**2.4 — Group and count statuses**
Group by class date, day name, topic, and batch name. Count `Present`, `Late`, and `Absent` records.

**2.5 — Filter and sort**
Keep classes with attendance records and order by class date.

In [0]:
--Write your code here
--Select class calendar columns From dim_class, select class date, class day, topic, and class key. Replace null topics with Topic not assigned.

select
    class_date,
    class_day,
    case
        when topic = 'nan' then 'Topic not assigned'
        else topic
        end as topic,
    class_key
from rivadataplatform.dataproduct.dim_class


In [0]:
--Write your code here
--Join attendance Join fact_attendance using class_key and add the attendance record identifier and status.
select
    c.class_key,
    a.attendance_id,
    a.attendance_status
from rivadataplatform.dataproduct.dim_class as c

join rivadataplatform.dataproduct.fact_attendance as a
    on c.class_key = a.class_key;   

In [0]:
--Add batch and date details Join dim_batch using batch_key and dim_date using date_key. Use the date dimension day name when available.
select
    b.batch_key,
    d.day_name,
    d.date_key,
    a.attendance_id,
    a.attendance_status
from rivadataplatform.dataproduct.fact_attendance as a

join rivadataplatform.dataproduct.dim_batch as b
    on a.batch_key = b.batch_key

join rivadataplatform.dataproduct.dim_date as d
    on a.date_key = d.date_key    



In [0]:
--Filter and sort Keep classes with attendance records and order by class date.
--Group and count statuses Group by class date, day name, topic, and batch name. Count Present, Late, and Absent records.
select
  c.class_date,
  d.day_name,
  c.topic,
  b.batch_name,

  count(case when a.attendance_status = 'Present' then 1 end) as present_count,
  count(case when a.attendance_status = 'Late' then 1 end) as late_count,
  count(case when a.attendance_status = 'Absent' then 1 end) as absent_count

from rivadataplatform.dataproduct.fact_attendance as a

join rivadataplatform.dataproduct.dim_class as c
    on a.class_key = c.class_key

join rivadataplatform.dataproduct.dim_batch as b
    on a.batch_key = b.batch_key

join rivadataplatform.dataproduct.dim_date as d
     on a.date_key = d.date_key

 group by
 c.class_date,
 d.day_name,
 c.topic,
 b.batch_name
 order by c.class_date DESC;  

## Question 3 — Null Profile and Attendance Check
Build the query in these steps:

**3.1 — Select student profile columns**
From `dim_student`, select student ID, student name, and phone number. Label blank or null phone numbers as `Phone missing`.

**3.2 — Add attendance**
Use a `LEFT JOIN` to connect `fact_attendance` through `student_key` and select the attendance record identifier.

**3.3 — Add class topics**
Use a `LEFT JOIN` to connect `dim_class` through `class_key` and inspect the topic.

**3.4 — Group and aggregate**
Group by student and phone status. Count attendance records and records with a missing topic.

**3.5 — Filter and sort**
Keep students with a missing phone or at least one missing topic, then order by student name.

In [0]:
--Write your code here
--Select student profile columns From dim_student, select student ID, student name, and phone number. Label blank or null phone numbers as Phone missing.
select
    student_id,
    student_name,
    case 
        when phone_no = 'nan'  then 'Phone missing'
        else phone_no
        end as phone_no
from rivadataplatform.dataproduct.dim_students

In [0]:
--Write your code here
--Add attendance Use a LEFT JOIN to connect fact_attendance through student_key and select the attendance record identifier.
select 
   s.student_id,
   s.student_name,
   s.student_key,
   a.attendance_id,
   a.attendance_status
from rivadataplatform.dataproduct.dim_students as s
left join rivadataplatform.dataproduct.fact_attendance as a
    on s.student_key = a.student_key;

In [0]:
--Add class topics Use a LEFT JOIN to connect dim_class through class_key and inspect the topic.
select
    c.class_key,
    c.topic 
    from rivadataplatform.dataproduct.dim_class as c 
    left join rivadataplatform.dataproduct.fact_attendance as a
    on c.class_key = a.class_key;

In [0]:
--Group and aggregate Group by student and phone status. Count attendance records and records with a missing topic.
-- Filter and sort Keep students with a missing phone or at least one missing topic, then order by student name.
select
    s.student_id,
    s.student_name,
    s.phone_no,
    case 
       when s.phone_no is null or s.phone_no = 'nan' then 'Phone missing'
       else 'phone available'
     end as phone_status,
     count (a.attendance_id) as attendance_count,
     count(
        case when c.topic = 'nan' then 1
        end
        ) as topic_missing
     from rivadataplatform.dataproduct.dim_students as s

     left join rivadataplatform.dataproduct.fact_attendance as a
        on s.student_key = a.student_key

     left join rivadataplatform.dataproduct.dim_class as c
        on a.class_key = c.class_key

     GROUP BY
    s.student_id,
    s.student_name,
    s.phone_no

HAVING
    s.phone_no IS NULL
    OR s.phone_no = ''
    OR s.phone_no = 'nan'
    OR COUNT(CASE WHEN c.topic = 'nan' THEN 1 END) >= 1

ORDER BY
    s.student_name;



## Question 4 — Batch Attendance Rate
Build the query in these steps:

**4.1 — Select batch columns**
From `dim_batch`, select `batch_id` and `batch_name`.

**4.2 — Join attendance**
Join `fact_attendance` using `batch_key` and add the attendance record identifier, student key, and status.

**4.3 — Group and count**
Group by batch ID and batch name. Count total records and distinct students.

**4.4 — Calculate the rate**
Count `Present` and `Late` as attended, divide by total records, multiply by 100, and use `NULLIF` to avoid division by zero.

**4.5 — Filter and sort**
Keep batches with at least one `Absent` record and order by attendance rate descending.

In [0]:
--Write your code here
--Select batch columns From dim_batch, select batch_id and batch_name.

select 
   batch_id,
   batch_name
 from rivadataplatform.dataproduct.dim_batch

In [0]:
--Write your code here
--Join attendance Join fact_attendance using batch_key and add the attendance record identifier, student key, and status.
----Group and count Group by batch ID and batch name. Count total records and distinct students.
----Calculate the rate Count Present and Late as attended, divide by total records, multiply by 100, and use NULLIF to avoid division by zero.
select 
   b.batch_id,
   b.batch_name,
   count (a.attendance_id) as total_records,
   count(distinct a.student_key) as distinct_students,
   count (case when a.attendance_status in ('Present', 'Late') then 1
    end
    ) * 100.0
   /NULLIF(count(a.attendance_id),0) as attendance_rate
from rivadataplatform.dataproduct.fact_attendance as a

join rivadataplatform.dataproduct.dim_batch as b
    on a.batch_key = b.batch_key

 

 group by 
    b.batch_id,
    b.batch_name

  having 
     count(
        case when a.attendance_status = 'absent' then 1
        end
     )  >=1

     order by 
        attendance_rate desc;



## Question 5 — Repeated Attendance by Student and Topic
Build the query in these steps:

**5.1 — Select student columns**
From `dim_student`, select student ID and student name.

**5.2 — Join attendance and classes**
Join `fact_attendance` using `student_key`, then join `dim_class` using `class_key`. Add the class date and topic.

**5.3 — Normalize the topic**
Replace null topic values with `Topic not assigned` so they form one grouping value.

**5.4 — Group and aggregate**
Group by student and topic. Count records, find the first and last class dates, and count `Late` or `Absent` records.

**5.5 — Filter and sort**
Keep student-topic groups with more than one record and order by attendance records descending, then student name.

In [0]:
--Write your code here
--Select student columns From dim_student, select student ID and student name.
select 
   student_id,
   student_name
   from rivadataplatform.dataproduct.dim_students
   

In [0]:
--Write your code here
--Join attendance and classes Join fact_attendance using student_key, then join dim_class using class_key. Add the class date and topic.
--Normalize the topic Replace null topic values with Topic not assigned so they form one grouping value.

--Group and aggregate Group by student and topic. Count records, find the first and last class dates, and count Late or Absent records.

--Filter and sort Keep student-topic groups with more than one record and order by attendance records descending, then student name.



select 
s.student_id,
s.student_name,

CASE
    WHEN c.topic IS NULL OR c.topic = 'nan'
        THEN 'Topic not assigned'
    ELSE c.topic
END AS topic,

count (a.attendance_id) as attendance_count,
min(c.class_date) as first_class_date,
max(c.class_date) as last_class_date,
count(case when a.attendance_status in ('Late', 'Absent') then 1
    end
    ) as late_absent_count

from rivadataplatform.dataproduct.dim_students as s


join rivadataplatform.dataproduct.fact_attendance as a
  on s.student_key = a.student_key 

join rivadataplatform.dataproduct.dim_class as c
  on c.class_key = a.class_key

GROUP BY
    s.student_id,
    s.student_name,
    CASE
        WHEN c.topic IS NULL OR c.topic = 'nan'
            THEN 'Topic not assigned'
        ELSE c.topic
    END

HAVING
    COUNT(a.attendance_id) > 1

ORDER BY
    attendance_count DESC,
    s.student_name ASC;      





## Question 6 — Student Present Coverage
Build the query in these steps:

**6.1 — Select student columns**
From `dim_student`, select student ID and student name.

**6.2 — Join attendance and batch**
Use `LEFT JOIN` to add `fact_attendance` through `student_key` and `dim_batch` through `batch_key`.

**6.3 — Group by student and batch**
Group by student ID, student name, and a null-safe batch label.

**6.4 — Calculate coverage metrics**
Count total attendance records, distinct classes, and `Present` records.

**6.5 — Sort the result**
Include students with zero attendance and order by present count descending, then student name.

In [0]:
--Write your code here
--Select student columns From dim_student, select student ID and student name.
select 
 student_id,
 student_name
 from rivadataplatform.dataproduct.dim_students

In [0]:
--Write your code here
--Join attendance and batch Use LEFT JOIN to add fact_attendance through student_key and dim_batch through batch_key.
--Group by student and batch Group by student ID, student name, and a null-safe batch label.
--Calculate coverage metrics Count total attendance records, distinct classes, and Present records.
--Sort the result Include students with zero attendance and order by present count descending, then student name.
select 

  s.student_id,
  s.student_name,
-- coalesce to replace null values with No batch
  coalesce(b.batch_name, 'No batch') as batch_label,

  count(a.attendance_id) as total_sttendance_records,
  count(distinct a.class_key) as distinct_classes,
  count(case when a.attendance_status = 'Present' then 1
    end
    ) as present_records




from rivadataplatform.dataproduct.dim_students as s

left join rivadataplatform.dataproduct.fact_attendance as a
  on s.student_key = a.student_key

 left join rivadataplatform.dataproduct.dim_batch as b
  on a.batch_key = b.batch_key 

  group by 
   s.student_id,
   s.student_name,
   coalesce(b.batch_name, 'No batch')


  order by 
   present_records desc,
   s.student_name asc;
